### Task1

### Import libraries

In [13]:
import pandas as pd
import numpy as np
import joblib
import json
import hashlib
import platform

from pathlib import Path
from datetime import datetime, timezone
from itertools import product

import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [14]:
df = pd.read_csv("heart_disease.csv")

display(df.head())
print(df.shape)
print(df.isnull().sum())
print(df["target"].value_counts())

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


(25, 14)
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64
target
1    16
0     9
Name: count, dtype: int64


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (20, 13)
Testing data: (5, 13)


In [16]:
df = df.drop_duplicates()
df = df.dropna(subset=["target"])

X = pd.get_dummies(df.drop(columns=["target"]), drop_first=False)
X = X.replace([np.inf, -np.inf], np.nan)

y = df["target"]

labels = sorted(y.unique())

if len(labels) != 2:
    raise ValueError("Binary classification target required")

if labels != [0, 1]:
    y = y.map({labels[0]: 0, labels[1]: 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [17]:
best_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))

Accuracy: 0.8
Precision: 0.75
Recall: 1.0
F1 Score: 0.8571428571428571


In [18]:
model_dir = Path("saved_models")
model_dir.mkdir(exist_ok=True)

version = datetime.now().strftime("v%Y%m%d_%H%M%S")

model_path = model_dir / f"heart_model_{version}.joblib"
metadata_path = model_dir / f"heart_model_{version}.json"

dataset_hash = hashlib.sha256(
    pd.util.hash_pandas_object(df, index=True).values.tobytes()
).hexdigest()

metadata = {
    "model_name": "Logistic Regression",
    "version": version,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "features": list(X_train.columns),
    "target_column": "target",
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "dataset_hash": dataset_hash,
    "metrics": {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_test, y_pred, zero_division=0))
    }
}

joblib.dump(best_model, model_path, compress=3)

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

print("Model saved:", model_path)
print("Metadata saved:", metadata_path)

Model saved: saved_models\heart_model_v20261013_160903.joblib
Metadata saved: saved_models\heart_model_v20261013_160903.json


In [19]:
loaded_model = joblib.load(model_path)

predictions = loaded_model.predict(X_test.iloc[:5])

print("Predictions:", predictions)
print("Actual values:", y_test.iloc[:5].to_numpy())

Predictions: [1 1 1 1 0]
Actual values: [1 1 0 1 0]


### Task2

In [20]:
base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

param_names = list(param_grid.keys())

combinations = list(
    product(*(param_grid[name] for name in param_names))
)

print("Total combinations:", len(combinations))

Total combinations: 27


In [21]:
grid_results = []

for values in combinations:
    params = dict(zip(param_names, values))
    model = clone(base_model)
    model.set_params(**params)

    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring="f1", n_jobs=-1
    )

    grid_results.append({
        "n_estimators": params["model__n_estimators"],
        "max_depth": params["model__max_depth"],
        "min_samples_leaf": params["model__min_samples_leaf"],
        "mean_f1": scores.mean(),
        "std_f1": scores.std()
    })

grid_results_df = pd.DataFrame(grid_results).sort_values(
    "mean_f1", ascending=False
).reset_index(drop=True)

display(grid_results_df.head(10))

,n_estimators,max_depth,min_samples_leaf,mean_f1,std_f1
0,200,NaN,1,0.893333,0.137275
1,200,10.0,1,0.893333,0.137275
2,200,5.0,1,0.893333,0.137275
3,100,10.0,4,0.853333,0.129271
4,200,NaN,2,0.853333,0.129271
5,200,10.0,2,0.853333,0.129271
6,200,5.0,4,0.853333,0.129271
7,200,NaN,4,0.853333,0.129271
8,200,5.0,2,0.853333,0.129271
9,100,5.0,4,0.853333,0.129271


In [22]:
best_row = grid_results_df.iloc[0]

best_params = {
    "model__n_estimators": int(best_row["n_estimators"]),
    "model__max_depth": None if pd.isna(best_row["max_depth"]) else int(best_row["max_depth"]),
    "model__min_samples_leaf": int(best_row["min_samples_leaf"])
}

grid_best_model = clone(base_model)
grid_best_model.set_params(**best_params)
grid_best_model.fit(X_train, y_train)

grid_pred = grid_best_model.predict(X_test)

print("Best parameters:", best_params)
print("Cross-validation F1:", best_row["mean_f1"])
print("Accuracy:", accuracy_score(y_test, grid_pred))
print("Precision:", precision_score(y_test, grid_pred, zero_division=0))
print("Recall:", recall_score(y_test, grid_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, grid_pred, zero_division=0))
print(classification_report(y_test, grid_pred, zero_division=0))

Best parameters: {'model__n_estimators': 200, 'model__max_depth': None, 'model__min_samples_leaf': 1}
Cross-validation F1: 0.8933333333333333
Accuracy: 0.8
Precision: 0.75
Recall: 1.0
F1 Score: 0.8571428571428571
              precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.75      1.00      0.86         3

    accuracy                           0.80         5
   macro avg       0.88      0.75      0.76         5
weighted avg       0.85      0.80      0.78         5



### Task 3 — Bayesian Optimization using Gaussian Process

In [23]:
search_space = [
    {
        "n_estimators": n,
        "max_depth": depth,
        "min_samples_leaf": leaf
    }
    for n in [100, 200, 300, 400]
    for depth in [None, 5, 10, 15]
    for leaf in [1, 2, 4]
]

print("Total combinations:", len(search_space))

Total combinations: 48


In [24]:
def encode_params(params):
    depth = 20 if params["max_depth"] is None else params["max_depth"]

    return np.array([
        params["n_estimators"] / 400,
        depth / 20,
        params["min_samples_leaf"] / 4
    ])

In [25]:
def evaluate_params(params):
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring="f1", n_jobs=-1
    )

    return scores.mean()

In [26]:
rng = np.random.default_rng(42)

initial_indices = rng.choice(
    len(search_space), size=5, replace=False
)

observed_indices = []
observed_X = []
observed_y = []

for idx in initial_indices:
    params = search_space[idx]
    score = evaluate_params(params)

    observed_indices.append(int(idx))
    observed_X.append(encode_params(params))
    observed_y.append(float(score))

    print(params, round(score, 4))

{'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 2} 0.8533
{'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 4} 0.8533
{'n_estimators': 300, 'max_depth': 10, 'min_samples_leaf': 1} 0.8267
{'n_estimators': 100, 'max_depth': 5, 'min_samples_leaf': 1} 0.8267
{'n_estimators': 400, 'max_depth': 15, 'min_samples_leaf': 4} 0.8533


In [27]:
kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=1.0, nu=2.5)
    + WhiteKernel(noise_level=0.001)
)

gp_model = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    random_state=42,
    n_restarts_optimizer=2
)

In [28]:
remaining_indices = [
    i for i in range(len(search_space))
    if i not in observed_indices
]

iterations = 8
beta = 1.5

for iteration in range(iterations):
    gp_model.fit(
        np.vstack(observed_X),
        np.array(observed_y)
    )

    candidate_X = np.vstack([
        encode_params(search_space[i])
        for i in remaining_indices
    ])

    mean_scores, uncertainty = gp_model.predict(
        candidate_X, return_std=True
    )

    ucb_scores = mean_scores + beta * uncertainty

    best_position = int(np.argmax(ucb_scores))
    chosen_index = remaining_indices.pop(best_position)

    params = search_space[chosen_index]
    score = evaluate_params(params)

    observed_indices.append(chosen_index)
    observed_X.append(encode_params(params))
    observed_y.append(float(score))

    print(
        f"Iteration {iteration + 1}:",
        params,
        "F1:", round(score, 4)
    )

Iteration 1: {'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1} F1: 0.8267
Iteration 2: {'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 2} F1: 0.8267


C:\Users\Bhagyashri\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Iteration 3: {'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 4} F1: 0.8533


C:\Users\Bhagyashri\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Iteration 4: {'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 2} F1: 0.8533


...

Iteration 6: {'n_estimators': 400, 'max_depth': 10, 'min_samples_leaf': 4} F1: 0.8533


C:\Users\Bhagyashri\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Iteration 7: {'n_estimators': 400, 'max_depth': 15, 'min_samples_leaf': 2} F1: 0.8533


C:\Users\Bhagyashri\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Iteration 8: {'n_estimators': 100, 'max_depth': 5, 'min_samples_leaf': 4} F1: 0.8533


In [29]:
best_position = int(np.argmax(observed_y))

bayes_best_params = search_space[
    observed_indices[best_position]
]

bayes_best_f1 = observed_y[best_position]

print("Best parameters:", bayes_best_params)
print("Best CV F1:", round(bayes_best_f1, 4))

Best parameters: {'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 2}
Best CV F1: 0.8533


In [30]:
bayes_best_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        **bayes_best_params,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

bayes_best_model.fit(X_train, y_train)

bayes_pred = bayes_best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, bayes_pred))
print("Precision:", precision_score(y_test, bayes_pred, zero_division=0))
print("Recall:", recall_score(y_test, bayes_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, bayes_pred, zero_division=0))
print(classification_report(y_test, bayes_pred, zero_division=0))

Accuracy: 0.8
Precision: 0.75
Recall: 1.0
F1 Score: 0.8571428571428571
              precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.75      1.00      0.86         3

    accuracy                           0.80         5
   macro avg       0.88      0.75      0.76         5
weighted avg       0.85      0.80      0.78         5

